<a href="https://colab.research.google.com/github/hilalozkan22/ConfidenceCalibration/blob/main/turkish_llm_confidence_pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA kullanılabilir:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU belleği:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB",
    )

PyTorch: 2.11.0+cu128
CUDA kullanılabilir: True
GPU: Tesla T4
GPU belleği: 14.56 GB


In [ ]:
!pip install -q -U \
    transformers \
    accelerate \
    bitsandbytes \
    sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 113.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00


In [ ]:
import torch
import numpy
import pandas
import transformers
import accelerate
import bitsandbytes
import sklearn

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

print("numpy:", numpy.__version__)
print("pandas:", pandas.__version__)
print("transformers:", transformers.__version__)
print("accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("scikit-learn:", sklearn.__version__)

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4
numpy: 2.0.2
pandas: 2.2.2
transformers: 5.13.1
accelerate: 1.14.0
bitsandbytes: 0.49.2
scikit-learn: 1.6.1


In [ ]:
import gc
import torch

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

MODEL_ID = "Qwen/Qwen3-4B"

# Önce olası eski GPU nesnelerini temizle.
gc.collect()
torch.cuda.empty_cache()

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("Tokenizer yükleniyor...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

print("Model yükleniyor...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=torch.float16,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)

model.eval()

print("\nModel başarıyla yüklendi.")
print("Model ID:", MODEL_ID)
print("Model cihazı:", model.device)
print("Tokenizer vocabulary:", len(tokenizer))

allocated_gb = torch.cuda.memory_allocated() / 1024**3
reserved_gb = torch.cuda.memory_reserved() / 1024**3

print(f"Kullanılan GPU belleği: {allocated_gb:.2f} GB")
print(f"Ayrılan GPU belleği: {reserved_gb:.2f} GB")

Tokenizer yükleniyor...


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]


Model başarıyla yüklendi.
Model ID: Qwen/Qwen3-4B
Model cihazı: cuda:0
Tokenizer vocabulary: 151669
Kullanılan GPU belleği: 2.49 GB
Ayrılan GPU belleği: 2.54 GB


In [ ]:
print("EOS token:", repr(tokenizer.eos_token))
print("EOS token ID:", tokenizer.eos_token_id)
print("PAD token:", repr(tokenizer.pad_token))
print("PAD token ID:", tokenizer.pad_token_id)
print("Chat template mevcut:", tokenizer.chat_template is not None)

EOS token: '<|im_end|>'
EOS token ID: 151645
PAD token: '<|endoftext|>'
PAD token ID: 151643
Chat template mevcut: True


In [ ]:
import torch


def build_answer_messages(question: str) -> list[dict[str, str]]:
    return [
        {
            "role": "system",
            "content": (
                "Sen kısa ve doğrudan cevap veren bir soru-cevap sistemisin. "
                "Uzun açıklama veya gerekçe üretme."
            ),
        },
        {
            "role": "user",
            "content": (
                "Aşağıdaki olgusal soruyu Türkçe ve mümkün olduğunca kısa cevapla.\n\n"
                "Kurallar:\n"
                "- Yalnızca cevabı yaz.\n"
                "- Açıklama veya gerekçe ekleme.\n"
                "- Soruyu tekrar etme.\n"
                '- Cevabı bilmiyorsan yalnızca "Bilmiyorum" yaz.\n\n'
                f"Soru: {question}\n\n"
                "Cevap:"
            ),
        },
    ]


@torch.inference_mode()
def generate_short_answer(
    question: str,
    max_new_tokens: int = 20,
) -> dict:
    messages = build_answer_messages(question)

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = outputs.sequences[0, input_length:]

    answer = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return {
        "question": question,
        "prompt_text": prompt_text,
        "answer": answer,
        "generated_ids": generated_ids.detach().cpu(),
        "scores": outputs.scores,
        "input_token_count": input_length,
        "output_token_count": len(generated_ids),
    }

In [ ]:
test_result = generate_short_answer(
    "Türkiye'nin başkenti neresidir?"
)

print("Soru:", test_result["question"])
print("Cevap:", repr(test_result["answer"]))
print("Girdi token sayısı:", test_result["input_token_count"])
print("Üretilen token sayısı:", test_result["output_token_count"])
print("Score adımı sayısı:", len(test_result["scores"]))
print("Üretilen token ID'leri:", test_result["generated_ids"].tolist())

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Soru: Türkiye'nin başkenti neresidir?
Cevap: 'Ankara.'
Girdi token sayısı: 146
Üretilen token sayısı: 5
Score adımı sayısı: 5
Üretilen token ID'leri: [2082, 74, 5059, 13, 151645]


In [ ]:
for position, token_id in enumerate(test_result["generated_ids"].tolist()):
    token_text = tokenizer.decode(
        [token_id],
        skip_special_tokens=False,
    )

    print(
        f"Pozisyon: {position} | "
        f"Token ID: {token_id} | "
        f"Token: {repr(token_text)} | "
        f"Özel token: {token_id in tokenizer.all_special_ids}"
    )

Pozisyon: 0 | Token ID: 2082 | Token: 'An' | Özel token: False
Pozisyon: 1 | Token ID: 74 | Token: 'k' | Özel token: False
Pozisyon: 2 | Token ID: 5059 | Token: 'ara' | Özel token: False
Pozisyon: 3 | Token ID: 13 | Token: '.' | Özel token: False
Pozisyon: 4 | Token ID: 151645 | Token: '<|im_end|>' | Özel token: True


In [ ]:
import math
import numpy as np
import torch.nn.functional as F


def extract_generated_token_logprobs(
    result: dict,
    exclude_special_tokens: bool = True,
) -> dict:
    generated_ids = result["generated_ids"]
    scores = result["scores"]

    if len(generated_ids) != len(scores):
        raise ValueError(
            f"Üretilen token sayısı ({len(generated_ids)}) ile "
            f"score sayısı ({len(scores)}) eşleşmiyor."
        )

    all_token_records = []
    content_token_records = []

    for position, (token_id_tensor, step_logits) in enumerate(
        zip(generated_ids, scores)
    ):
        token_id = int(token_id_tensor.item())
        is_special = token_id in tokenizer.all_special_ids

        step_log_probs = F.log_softmax(
            step_logits[0].float(),
            dim=-1,
        )

        token_logprob = float(step_log_probs[token_id].item())
        token_probability = float(math.exp(token_logprob))

        record = {
            "position": position,
            "token_id": token_id,
            "token": tokenizer.decode(
                [token_id],
                skip_special_tokens=False,
            ),
            "is_special": is_special,
            "logprob": token_logprob,
            "probability": token_probability,
        }

        all_token_records.append(record)

        if not (exclude_special_tokens and is_special):
            content_token_records.append(record)

    if not content_token_records:
        raise ValueError(
            "Özel tokenlar çıkarıldıktan sonra ölçülebilecek cevap tokenı kalmadı."
        )

    logprob_values = [
        record["logprob"]
        for record in content_token_records
    ]

    return {
        "all_tokens": all_token_records,
        "content_tokens": content_token_records,
        "content_token_count": len(content_token_records),
        "mean_logprob": float(np.mean(logprob_values)),
        "min_logprob": float(np.min(logprob_values)),
        "max_logprob": float(np.max(logprob_values)),
        "sequence_logprob": float(np.sum(logprob_values)),
        "geometric_mean_probability": float(
            math.exp(np.mean(logprob_values))
        ),
    }

In [ ]:
logprob_result = extract_generated_token_logprobs(test_result)

print("Cevap:", repr(test_result["answer"]))
print("Metinsel token sayısı:", logprob_result["content_token_count"])
print("Mean logprob:", logprob_result["mean_logprob"])
print("Min logprob:", logprob_result["min_logprob"])
print("Sequence logprob:", logprob_result["sequence_logprob"])
print(
    "Geometrik ortalama token olasılığı:",
    logprob_result["geometric_mean_probability"],
)

print("\nTüm üretilen tokenlar:")
for record in logprob_result["all_tokens"]:
    print(record)

print("\nConfidence hesabına giren tokenlar:")
for record in logprob_result["content_tokens"]:
    print(record)

Cevap: 'Ankara.'
Metinsel token sayısı: 4
Mean logprob: -0.014461868111538934
Min logprob: -0.05516854673624039
Sequence logprob: -0.057847472446155734
Geometrik ortalama token olasılığı: 0.9856422024143344

Tüm üretilen tokenlar:
{'position': 0, 'token_id': 2082, 'token': 'An', 'is_special': False, 'logprob': -0.05516854673624039, 'probability': 0.9463256344140172}
{'position': 1, 'token_id': 74, 'token': 'k', 'is_special': False, 'logprob': -0.00041607304592616856, 'probability': 0.99958401350046}
{'position': 2, 'token_id': 5059, 'token': 'ara', 'is_special': False, 'logprob': -3.6238969187252223e-05, 'probability': 0.9999637616874363}
{'position': 3, 'token_id': 13, 'token': '.', 'is_special': False, 'logprob': -0.0022266136948019266, 'probability': 0.9977758633706406}
{'position': 4, 'token_id': 151645, 'token': '<|im_end|>', 'is_special': True, 'logprob': -3.576272320060525e-06, 'probability': 0.9999964237340748}

Confidence hesabına giren tokenlar:
{'position': 0, 'token_id': 20

In [ ]:
import re
from typing import Optional


def build_confidence_messages(
    question: str,
    answer: str,
) -> list[dict[str, str]]:
    return [
        {
            "role": "system",
            "content": (
                "Sen verilen bir cevabın doğruluğunu değerlendiren bir sistemsin. "
                "İstenen çıktı formatına kesinlikle uy."
            ),
        },
        {
            "role": "user",
            "content": (
                "Aşağıda bir soru ve bu soruya verilmiş bir cevap bulunmaktadır.\n\n"
                f"Soru: {question}\n"
                f"Verilen cevap: {answer}\n\n"
                "Verilen cevabın tamamen doğru olma olasılığını "
                "0 ile 100 arasında değerlendir.\n\n"
                "Kurallar:\n"
                "- Yalnızca bir tam sayı yaz.\n"
                "- Yüzde işareti kullanma.\n"
                "- Açıklama veya gerekçe yazma.\n\n"
                "Güven:"
            ),
        },
    ]


@torch.inference_mode()
def generate_text_from_messages(
    messages: list[dict[str, str]],
    max_new_tokens: int = 10,
) -> str:
    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = outputs[0, input_length:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()


def parse_confidence(raw_output: str) -> Optional[int]:
    text = raw_output.strip()

    # İdeal biçim: yalnızca 0-100 arasında tam sayı.
    if re.fullmatch(r"\d{1,3}", text):
        value = int(text)
        return value if 0 <= value <= 100 else None

    # Prompt ihlal edilse bile tek bir sayı varsa kurtarmayı dene.
    matches = re.findall(r"(?<!\d)(100|\d{1,2})(?!\d)", text)

    if len(matches) != 1:
        return None

    value = int(matches[0])
    return value if 0 <= value <= 100 else None

In [ ]:
confidence_raw = generate_text_from_messages(
    build_confidence_messages(
        test_result["question"],
        test_result["answer"],
    ),
    max_new_tokens=10,
)

confidence_value = parse_confidence(confidence_raw)

strict_format_compliance = bool(
    re.fullmatch(r"\d{1,3}", confidence_raw.strip())
)

print("Confidence ham çıktı:", repr(confidence_raw))
print("Parse edilen confidence:", confidence_value)
print("Tam format uyumu:", strict_format_compliance)

Confidence ham çıktı: '100'
Parse edilen confidence: 100
Tam format uyumu: True


In [ ]:
wrong_answer = "İstanbul"

wrong_confidence_raw = generate_text_from_messages(
    build_confidence_messages(
        "Türkiye'nin başkenti neresidir?",
        wrong_answer,
    ),
    max_new_tokens=10,
)

wrong_confidence_value = parse_confidence(wrong_confidence_raw)

wrong_strict_format_compliance = bool(
    re.fullmatch(r"\d{1,3}", wrong_confidence_raw.strip())
)

print("Yanlış cevap:", wrong_answer)
print("Confidence ham çıktı:", repr(wrong_confidence_raw))
print("Parse edilen confidence:", wrong_confidence_value)
print("Tam format uyumu:", wrong_strict_format_compliance)

Yanlış cevap: İstanbul
Confidence ham çıktı: '100'
Parse edilen confidence: 100
Tam format uyumu: True


In [ ]:
def build_verification_messages(
    question: str,
    answer: str,
) -> list[dict[str, str]]:
    return [
        {
            "role": "system",
            "content": (
                "Sen olgusal cevapları dikkatle değerlendiren bir doğrulama sistemisin. "
                "Verilen cevap yanlışsa bunu açıkça belirt. "
                "Yalnızca izin verilen seçeneği üret."
            ),
        },
        {
            "role": "user",
            "content": (
                "Aşağıda bir soru ve bu soruya verilmiş bir cevap bulunmaktadır.\n\n"
                f"Soru: {question}\n"
                f"Verilen cevap: {answer}\n\n"
                "Bu cevap olgusal olarak tamamen doğru mudur?\n\n"
                "A) Doğru\n"
                "B) Yanlış\n"
                "C) Emin değilim\n\n"
                "Yalnızca A, B veya C yaz."
            ),
        },
    ]


def parse_verification(raw_output: str) -> str | None:
    text = raw_output.strip().upper()

    if text in {"A", "B", "C"}:
        return text

    match = re.search(r"\b([ABC])\b", text)
    return match.group(1) if match else None

In [ ]:
correct_verification_raw = generate_text_from_messages(
    build_verification_messages(
        "Türkiye'nin başkenti neresidir?",
        "Ankara",
    ),
    max_new_tokens=5,
)

correct_verification = parse_verification(
    correct_verification_raw
)

print("Doğru cevap testi")
print("Ham çıktı:", repr(correct_verification_raw))
print("Parse edilen seçim:", correct_verification)

Doğru cevap testi
Ham çıktı: 'A'
Parse edilen seçim: A


In [ ]:
wrong_verification_raw = generate_text_from_messages(
    build_verification_messages(
        "Türkiye'nin başkenti neresidir?",
        "İstanbul",
    ),
    max_new_tokens=5,
)

wrong_verification = parse_verification(
    wrong_verification_raw
)

print("Yanlış cevap testi")
print("Ham çıktı:", repr(wrong_verification_raw))
print("Parse edilen seçim:", wrong_verification)

Yanlış cevap testi
Ham çıktı: 'A'
Parse edilen seçim: A


In [ ]:
import torch
import torch.nn.functional as F


def get_single_token_id(text: str) -> int:
    token_ids = tokenizer.encode(
        text,
        add_special_tokens=False,
    )

    print(f"{repr(text)} tokenları:", token_ids)

    if len(token_ids) != 1:
        raise ValueError(
            f"{repr(text)} tek token değil: {token_ids}"
        )

    return token_ids[0]


# Hem boşluksuz hem boşluklu biçimleri kontrol ediyoruz.
for candidate in ["A", "B", "C", " A", " B", " C"]:
    print(
        repr(candidate),
        tokenizer.encode(candidate, add_special_tokens=False),
    )

'A' [32]
'B' [33]
'C' [34]
' A' [362]
' B' [425]
' C' [356]


In [ ]:
@torch.inference_mode()
def get_verification_probabilities(
    question: str,
    answer: str,
) -> dict:
    messages = build_verification_messages(
        question,
        answer,
    )

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(model.device)

    outputs = model(**inputs)

    # Prompttan sonra üretilecek ilk tokenın logits değerleri.
    next_token_logits = outputs.logits[0, -1].float()
    next_token_probs = F.softmax(next_token_logits, dim=-1)

    candidates = {}

    for label in ["A", "B", "C"]:
        token_variants = {
            label: tokenizer.encode(
                label,
                add_special_tokens=False,
            ),
            f" {label}": tokenizer.encode(
                f" {label}",
                add_special_tokens=False,
            ),
        }

        valid_variants = {}

        for variant_text, token_ids in token_variants.items():
            if len(token_ids) == 1:
                token_id = token_ids[0]
                valid_variants[variant_text] = {
                    "token_id": token_id,
                    "probability": float(
                        next_token_probs[token_id].item()
                    ),
                    "logit": float(
                        next_token_logits[token_id].item()
                    ),
                }

        if not valid_variants:
            raise ValueError(
                f"{label} için tek tokenlı bir biçim bulunamadı."
            )

        # Olasılığı en yüksek geçerli yazım biçimini kullan.
        selected_variant = max(
            valid_variants,
            key=lambda key: valid_variants[key]["probability"],
        )

        candidates[label] = {
            "selected_variant": selected_variant,
            **valid_variants[selected_variant],
            "all_variants": valid_variants,
        }

    raw_probabilities = {
        label: candidates[label]["probability"]
        for label in ["A", "B", "C"]
    }

    total = sum(raw_probabilities.values())

    # Yalnızca A/B/C seçenekleri içerisinde yeniden normalize ediyoruz.
    normalized_probabilities = {
        label: probability / total
        for label, probability in raw_probabilities.items()
    }

    predicted_label = max(
        normalized_probabilities,
        key=normalized_probabilities.get,
    )

    return {
        "raw_probabilities": raw_probabilities,
        "normalized_probabilities": normalized_probabilities,
        "predicted_label": predicted_label,
        "candidate_details": candidates,
    }

In [ ]:
correct_probs = get_verification_probabilities(
    "Türkiye'nin başkenti neresidir?",
    "Ankara",
)

wrong_probs = get_verification_probabilities(
    "Türkiye'nin başkenti neresidir?",
    "İstanbul",
)

print("DOĞRU CEVAP — ANKARA")
print(correct_probs["normalized_probabilities"])
print("Tahmin:", correct_probs["predicted_label"])

print("\nYANLIŞ CEVAP — İSTANBUL")
print(wrong_probs["normalized_probabilities"])
print("Tahmin:", wrong_probs["predicted_label"])

DOĞRU CEVAP — ANKARA
{'A': 0.9999999987003241, 'B': 6.092758512008022e-10, 'C': 6.904000140381809e-10}
Tahmin: A

YANLIŞ CEVAP — İSTANBUL
{'A': 0.9999999918365111, 'B': 5.346226084822611e-09, 'C': 2.817262844439636e-09}
Tahmin: A


In [ ]:
def build_reversed_verification_messages(
    question: str,
    answer: str,
) -> list[dict[str, str]]:
    return [
        {
            "role": "system",
            "content": (
                "Sen olgusal cevapları değerlendiren bir doğrulama sistemisin. "
                "Cevabı dikkatle kontrol et ve yalnızca bir seçenek üret."
            ),
        },
        {
            "role": "user",
            "content": (
                "Aşağıda bir soru ve bu soruya verilmiş bir cevap bulunmaktadır.\n\n"
                f"Soru: {question}\n"
                f"Verilen cevap: {answer}\n\n"
                "Bu cevap olgusal olarak tamamen doğru mudur?\n\n"
                "A) Yanlış\n"
                "B) Doğru\n"
                "C) Emin değilim\n\n"
                "Yalnızca A, B veya C yaz."
            ),
        },
    ]

In [ ]:
reversed_correct_raw = generate_text_from_messages(
    build_reversed_verification_messages(
        "Türkiye'nin başkenti neresidir?",
        "Ankara",
    ),
    max_new_tokens=5,
)

reversed_wrong_raw = generate_text_from_messages(
    build_reversed_verification_messages(
        "Türkiye'nin başkenti neresidir?",
        "İstanbul",
    ),
    max_new_tokens=5,
)

print("Ters sıra — doğru cevap:", repr(reversed_correct_raw))
print("Ters sıra — yanlış cevap:", repr(reversed_wrong_raw))

Ters sıra — doğru cevap: 'B'
Ters sıra — yanlış cevap: 'B'


In [ ]:
sanity_checks = [
    {
        "question": "Türkiye'nin başkenti neresidir?",
        "answer": "Ankara",
        "expected": "A",
        "case": "correct",
    },
    {
        "question": "Türkiye'nin başkenti neresidir?",
        "answer": "İstanbul",
        "expected": "B",
        "case": "wrong",
    },
    {
        "question": "Suyun kimyasal formülü nedir?",
        "answer": "H2O",
        "expected": "A",
        "case": "correct",
    },
    {
        "question": "Suyun kimyasal formülü nedir?",
        "answer": "CO2",
        "expected": "B",
        "case": "wrong",
    },
    {
        "question": "Bir üçgenin iç açılarının toplamı kaç derecedir?",
        "answer": "180",
        "expected": "A",
        "case": "correct",
    },
    {
        "question": "Bir üçgenin iç açılarının toplamı kaç derecedir?",
        "answer": "360",
        "expected": "B",
        "case": "wrong",
    },
    {
        "question": "Dünya'nın doğal uydusunun adı nedir?",
        "answer": "Ay",
        "expected": "A",
        "case": "correct",
    },
    {
        "question": "Dünya'nın doğal uydusunun adı nedir?",
        "answer": "Mars",
        "expected": "B",
        "case": "wrong",
    },
]

In [ ]:
sanity_results = []

for item in sanity_checks:
    raw_output = generate_text_from_messages(
        build_verification_messages(
            item["question"],
            item["answer"],
        ),
        max_new_tokens=5,
    )

    parsed = parse_verification(raw_output)

    probability_result = get_verification_probabilities(
        item["question"],
        item["answer"],
    )

    row = {
        **item,
        "raw_output": raw_output,
        "parsed": parsed,
        "p_A": probability_result[
            "normalized_probabilities"
        ]["A"],
        "p_B": probability_result[
            "normalized_probabilities"
        ]["B"],
        "p_C": probability_result[
            "normalized_probabilities"
        ]["C"],
        "probability_prediction": probability_result[
            "predicted_label"
        ],
    }

    sanity_results.append(row)

import pandas as pd

sanity_df = pd.DataFrame(sanity_results)
sanity_df

,question,answer,expected,case,raw_output,parsed,p_A,p_B,p_C,probability_prediction
0,Türkiye'nin başkenti neresidir?,Ankara,A,correct,A,A,1.000000e+00,6.092759e-10,6.904000e-10,A
1,Türkiye'nin başkenti neresidir?,İstanbul,B,wrong,A,A,1.000000e+00,5.346226e-09,2.817263e-09,A
2,Suyun kimyasal formülü nedir?,H2O,A,correct,A,A,1.000000e+00,2.138759e-10,4.972764e-10,A
3,Suyun kimyasal formülü nedir?,CO2,B,wrong,B,B,1.958514e-11,1.000000e+00,4.972764e-10,B
4,Bir üçgenin iç açılarının toplamı kaç derecedir?,180,A,correct,A,A,1.000000e+00,1.373024e-09,7.465004e-10,A
5,Bir üçgenin iç açılarının toplamı kaç derecedir?,360,B,wrong,B,B,3.437299e-11,1.000000e+00,3.581748e-10,B
6,Dünya'nın doğal uydusunun adı nedir?,Ay,A,correct,A,A,9.987917e-01,1.206588e-03,1.758219e-06,A
7,Dünya'nın doğal uydusunun adı nedir?,Mars,B,wrong,B,B,1.402633e-10,1.000000e+00,5.515933e-09,B


In [ ]:
pilot_questions = [
    {
        "question_id": "Q01",
        "question": "Türkiye'nin başkenti neresidir?",
        "reference_answer": "Ankara",
        "accepted_aliases": ["Ankara"],
        "answer_type": "LOCATION",
        "difficulty": "easy",
    },
    {
        "question_id": "Q02",
        "question": "Türkiye Cumhuriyeti hangi yıl ilan edildi?",
        "reference_answer": "1923",
        "accepted_aliases": ["1923", "29 Ekim 1923"],
        "answer_type": "DATE",
        "difficulty": "easy",
    },
    {
        "question_id": "Q03",
        "question": "Dünya'nın doğal uydusunun adı nedir?",
        "reference_answer": "Ay",
        "accepted_aliases": ["Ay"],
        "answer_type": "OBJECT",
        "difficulty": "easy",
    },
    {
        "question_id": "Q04",
        "question": "Ay'a ilk ayak basan insan kimdir?",
        "reference_answer": "Neil Armstrong",
        "accepted_aliases": [
            "Neil Armstrong",
            "Neil Alden Armstrong",
        ],
        "answer_type": "PERSON",
        "difficulty": "easy",
    },
    {
        "question_id": "Q05",
        "question": "Suyun kimyasal formülü nedir?",
        "reference_answer": "H2O",
        "accepted_aliases": ["H2O", "H₂O"],
        "answer_type": "SCIENCE",
        "difficulty": "easy",
    },
    {
        "question_id": "Q06",
        "question": "İstiklal Marşı'nın yazarı kimdir?",
        "reference_answer": "Mehmet Akif Ersoy",
        "accepted_aliases": [
            "Mehmet Akif Ersoy",
            "Mehmet Âkif Ersoy",
        ],
        "answer_type": "PERSON",
        "difficulty": "easy",
    },
    {
        "question_id": "Q07",
        "question": "Bir üçgenin iç açılarının toplamı kaç derecedir?",
        "reference_answer": "180",
        "accepted_aliases": ["180", "180 derece"],
        "answer_type": "NUMBER",
        "difficulty": "easy",
    },
    {
        "question_id": "Q08",
        "question": "Güneş Sistemi'nin en büyük gezegeni hangisidir?",
        "reference_answer": "Jüpiter",
        "accepted_aliases": ["Jüpiter"],
        "answer_type": "OBJECT",
        "difficulty": "easy",
    },
    {
        "question_id": "Q09",
        "question": "Türkiye'nin en kalabalık şehri hangisidir?",
        "reference_answer": "İstanbul",
        "accepted_aliases": ["İstanbul"],
        "answer_type": "LOCATION",
        "difficulty": "easy",
    },
    {
        "question_id": "Q10",
        "question": "DNA'nın Türkçe açılımı nedir?",
        "reference_answer": "Deoksiribonükleik asit",
        "accepted_aliases": [
            "Deoksiribonükleik asit",
            "Deoksiribonükleik asit molekülü",
        ],
        "answer_type": "CONCEPT",
        "difficulty": "easy",
    },
    {
        "question_id": "Q11",
        "question": "Osmanlı Devleti'nin kurucusu kimdir?",
        "reference_answer": "Osman Gazi",
        "accepted_aliases": [
            "Osman Gazi",
            "I. Osman",
            "Osman Bey",
        ],
        "answer_type": "PERSON",
        "difficulty": "medium",
    },
    {
        "question_id": "Q12",
        "question": "Lozan Barış Antlaşması hangi yıl imzalandı?",
        "reference_answer": "1923",
        "accepted_aliases": ["1923", "24 Temmuz 1923"],
        "answer_type": "DATE",
        "difficulty": "medium",
    },
    {
        "question_id": "Q13",
        "question": "Periyodik tabloda atom numarası 8 olan element hangisidir?",
        "reference_answer": "Oksijen",
        "accepted_aliases": ["Oksijen", "O"],
        "answer_type": "SCIENCE",
        "difficulty": "medium",
    },
    {
        "question_id": "Q14",
        "question": "Suç ve Ceza romanının yazarı kimdir?",
        "reference_answer": "Fyodor Dostoyevski",
        "accepted_aliases": [
            "Fyodor Dostoyevski",
            "Dostoyevski",
            "Fyodor Mihayloviç Dostoyevski",
        ],
        "answer_type": "PERSON",
        "difficulty": "medium",
    },
    {
        "question_id": "Q15",
        "question": "Işık boşlukta yaklaşık saniyede kaç kilometre yol alır?",
        "reference_answer": "300000",
        "accepted_aliases": [
            "300000",
            "300.000",
            "299792",
            "299.792",
            "300000 km",
            "300.000 km",
        ],
        "answer_type": "NUMBER",
        "difficulty": "medium",
    },
    {
        "question_id": "Q16",
        "question": "Magna Carta hangi ülkede imzalanmıştır?",
        "reference_answer": "İngiltere",
        "accepted_aliases": [
            "İngiltere",
            "England",
        ],
        "answer_type": "LOCATION",
        "difficulty": "medium",
    },
    {
        "question_id": "Q17",
        "question": "TCP kısaltmasının açılımı nedir?",
        "reference_answer": "Transmission Control Protocol",
        "accepted_aliases": [
            "Transmission Control Protocol",
            "İletim Kontrol Protokolü",
        ],
        "answer_type": "CONCEPT",
        "difficulty": "medium",
    },
    {
        "question_id": "Q18",
        "question": "Python programlama dilinin yaratıcısı kimdir?",
        "reference_answer": "Guido van Rossum",
        "accepted_aliases": [
            "Guido van Rossum",
            "Guido Van Rossum",
        ],
        "answer_type": "PERSON",
        "difficulty": "medium",
    },
    {
        "question_id": "Q19",
        "question": "Fotosentez sırasında bitkiler atmosferden hangi gazı alır?",
        "reference_answer": "Karbondioksit",
        "accepted_aliases": [
            "Karbondioksit",
            "CO2",
            "CO₂",
        ],
        "answer_type": "SCIENCE",
        "difficulty": "medium",
    },
    {
        "question_id": "Q20",
        "question": "İnsan vücudundaki en büyük organ hangisidir?",
        "reference_answer": "Deri",
        "accepted_aliases": [
            "Deri",
            "Cilt",
        ],
        "answer_type": "SCIENCE",
        "difficulty": "medium",
    },
]

print("Toplam soru:", len(pilot_questions))

Toplam soru: 20


In [ ]:
import os
import re
import time
import unicodedata
import pandas as pd


MINI_PILOT_PATH = "/content/qwen3_4b_turkish_confidence_first5.csv"


def normalize_text(text: str) -> str:
    """
    Basit pilot normalizasyonu.
    Türkçe karakterleri korur; noktalama ve fazla boşlukları temizler.
    """
    text = unicodedata.normalize("NFKC", str(text))
    text = text.casefold().strip()
    text = re.sub(r"[^\w\s]", " ", text, flags=re.UNICODE)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def is_exact_or_alias_match(
    generated_answer: str,
    accepted_aliases: list[str],
) -> bool:
    normalized_generated = normalize_text(generated_answer)

    normalized_aliases = {
        normalize_text(alias)
        for alias in accepted_aliases
    }

    return normalized_generated in normalized_aliases


def is_short_answer_format(answer: str) -> bool:
    """
    Pilot için kaba format kontrolü.
    Doğruluk değerlendirmesinden ayrıdır.
    """
    stripped = answer.strip()

    if not stripped:
        return False

    line_count = len(
        [line for line in stripped.splitlines() if line.strip()]
    )
    word_count = len(stripped.split())

    return line_count == 1 and word_count <= 12


def parse_verification_strict(raw_output: str) -> str | None:
    text = raw_output.strip().upper()

    if text in {"A", "B", "C"}:
        return text

    match = re.search(r"\b([ABC])\b", text)
    return match.group(1) if match else None

# Tüm Feature'lar Pilot Test

In [ ]:
def run_single_pilot_question(item: dict) -> dict:
    question = item["question"]

    start_time = time.time()

    # 1. Ana cevap
    answer_result = generate_short_answer(
        question=question,
        max_new_tokens=20,
    )

    generated_answer = answer_result["answer"]

    # 2. Token log-probability
    logprob_result = extract_generated_token_logprobs(
        answer_result,
        exclude_special_tokens=True,
    )

    # 3. Sayısal self-confidence
    confidence_raw = generate_text_from_messages(
        build_confidence_messages(
            question=question,
            answer=generated_answer,
        ),
        max_new_tokens=10,
    )

    confidence_value = parse_confidence(confidence_raw)

    confidence_format_compliant = bool(
        re.fullmatch(r"\d{1,3}", confidence_raw.strip())
    )

    # 4. A/B/C self-verification
    verification_raw = generate_text_from_messages(
        build_verification_messages(
            question=question,
            answer=generated_answer,
        ),
        max_new_tokens=5,
    )

    verification_label = parse_verification_strict(
        verification_raw
    )

    # 5. A/B/C token olasılıkları
    verification_probs = get_verification_probabilities(
        question=question,
        answer=generated_answer,
    )

    normalized_verification_probs = (
        verification_probs["normalized_probabilities"]
    )

    # 6. Basit otomatik eşleşme
    automatic_alias_match = is_exact_or_alias_match(
        generated_answer=generated_answer,
        accepted_aliases=item["accepted_aliases"],
    )

    elapsed_seconds = time.time() - start_time

    return {
        "question_id": item["question_id"],
        "question": question,
        "reference_answer": item["reference_answer"],
        "accepted_aliases": " | ".join(
            item["accepted_aliases"]
        ),
        "answer_type": item["answer_type"],
        "difficulty": item["difficulty"],

        "model_id": MODEL_ID,
        "quantization": "4bit_nf4",
        "answer_prompt_version": "answer_v1",
        "confidence_prompt_version": "confidence_v1",
        "verification_prompt_version": "verification_v1",

        "generated_answer": generated_answer,
        "normalized_answer": normalize_text(
            generated_answer
        ),

        # Bunlar daha sonra elle kontrol edilecek.
        "manual_is_correct": None,
        "manual_error_type": "",
        "manual_notes": "",

        # İlk otomatik gösterge.
        "automatic_alias_match": int(
            automatic_alias_match
        ),

        "answer_format_compliant": int(
            is_short_answer_format(generated_answer)
        ),

        "input_token_count": (
            answer_result["input_token_count"]
        ),
        "output_token_count_all": (
            answer_result["output_token_count"]
        ),
        "output_content_token_count": (
            logprob_result["content_token_count"]
        ),

        "mean_logprob": (
            logprob_result["mean_logprob"]
        ),
        "min_logprob": (
            logprob_result["min_logprob"]
        ),
        "max_logprob": (
            logprob_result["max_logprob"]
        ),
        "sequence_logprob": (
            logprob_result["sequence_logprob"]
        ),
        "geometric_mean_probability": (
            logprob_result[
                "geometric_mean_probability"
            ]
        ),

        "self_confidence_raw": confidence_raw,
        "self_confidence_value": confidence_value,
        "confidence_parse_success": int(
            confidence_value is not None
        ),
        "confidence_format_compliant": int(
            confidence_format_compliant
        ),

        "verification_raw": verification_raw,
        "verification_label": verification_label,
        "verification_parse_success": int(
            verification_label is not None
        ),

        "verification_p_A": (
            normalized_verification_probs["A"]
        ),
        "verification_p_B": (
            normalized_verification_probs["B"]
        ),
        "verification_p_C": (
            normalized_verification_probs["C"]
        ),
        "verification_probability_prediction": (
            verification_probs["predicted_label"]
        ),

        "elapsed_seconds": elapsed_seconds,
    }

In [ ]:
single_pipeline_test = run_single_pilot_question(
    pilot_questions[0]
)

single_pipeline_test

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


{'question_id': 'Q01',
 'question': "Türkiye'nin başkenti neresidir?",
 'reference_answer': 'Ankara',
 'accepted_aliases': 'Ankara',
 'answer_type': 'LOCATION',
 'difficulty': 'easy',
 'model_id': 'Qwen/Qwen3-4B',
 'quantization': '4bit_nf4',
 'answer_prompt_version': 'answer_v1',
 'confidence_prompt_version': 'confidence_v1',
 'verification_prompt_version': 'verification_v1',
 'generated_answer': 'Ankara.',
 'normalized_answer': 'ankara',
 'manual_is_correct': None,
 'manual_error_type': '',
 'manual_notes': '',
 'automatic_alias_match': 1,
 'answer_format_compliant': 1,
 'input_token_count': 146,
 'output_token_count_all': 5,
 'output_content_token_count': 4,
 'mean_logprob': -0.014461868111538934,
 'min_logprob': -0.05516854673624039,
 'max_logprob': -3.6238969187252223e-05,
 'sequence_logprob': -0.057847472446155734,
 'geometric_mean_probability': 0.9856422024143344,
 'self_confidence_raw': '100',
 'self_confidence_value': 100,
 'confidence_parse_success': 1,
 'confidence_format_co

In [ ]:
mini_results = []

for index, item in enumerate(
    pilot_questions[:5],
    start=1,
):
    print(
        f"[{index}/5] Çalıştırılıyor: "
        f"{item['question_id']} — "
        f"{item['question']}"
    )

    try:
        result = run_single_pilot_question(item)
        mini_results.append(result)

        current_df = pd.DataFrame(mini_results)

        current_df.to_csv(
            MINI_PILOT_PATH,
            index=False,
            encoding="utf-8-sig",
        )

        print(
            "  Cevap:",
            repr(result["generated_answer"]),
        )
        print(
            "  Alias eşleşmesi:",
            result["automatic_alias_match"],
        )
        print(
            "  Mean logprob:",
            round(result["mean_logprob"], 6),
        )
        print(
            "  Self-confidence:",
            result["self_confidence_value"],
        )
        print(
            "  Verification:",
            result["verification_label"],
        )
        print(
            "  Süre:",
            round(result["elapsed_seconds"], 2),
            "sn",
        )
        print()

    except Exception as error:
        print(
            f"  HATA — {item['question_id']}: "
            f"{type(error).__name__}: {error}"
        )

        # Önceki başarılı sonuçlar yine kaydedilir.
        if mini_results:
            pd.DataFrame(mini_results).to_csv(
                MINI_PILOT_PATH,
                index=False,
                encoding="utf-8-sig",
            )

        raise

print("Mini pilot tamamlandı.")
print("CSV:", MINI_PILOT_PATH)

[1/5] Çalıştırılıyor: Q01 — Türkiye'nin başkenti neresidir?
  Cevap: 'Ankara.'
  Alias eşleşmesi: 1
  Mean logprob: -0.014462
  Self-confidence: 100
  Verification: A
  Süre: 1.5 sn

[2/5] Çalıştırılıyor: Q02 — Türkiye Cumhuriyeti hangi yıl ilan edildi?
  Cevap: '1923.'
  Alias eşleşmesi: 1
  Mean logprob: -0.121519
  Self-confidence: 90
  Verification: A
  Süre: 1.47 sn

[3/5] Çalıştırılıyor: Q03 — Dünya'nın doğal uydusunun adı nedir?
  Cevap: 'Bilmiyorum.'
  Alias eşleşmesi: 0
  Mean logprob: -0.000431
  Self-confidence: 50
  Verification: B
  Süre: 1.85 sn

[4/5] Çalıştırılıyor: Q04 — Ay'a ilk ayak basan insan kimdir?
  Cevap: 'Neil Armstrong.'
  Alias eşleşmesi: 1
  Mean logprob: -0.035184
  Self-confidence: 85
  Verification: A
  Süre: 1.49 sn

[5/5] Çalıştırılıyor: Q05 — Suyun kimyasal formülü nedir?
  Cevap: 'H₂O'
  Alias eşleşmesi: 1
  Mean logprob: -0.00119
  Self-confidence: 95
  Verification: A
  Süre: 1.62 sn

Mini pilot tamamlandı.
CSV: /content/qwen3_4b_turkish_confidence

In [ ]:
mini_df = pd.read_csv(MINI_PILOT_PATH)

display_columns = [
    "question_id",
    "question",
    "reference_answer",
    "generated_answer",
    "automatic_alias_match",
    "answer_format_compliant",
    "mean_logprob",
    "geometric_mean_probability",
    "self_confidence_value",
    "verification_label",
    "verification_p_A",
    "verification_p_B",
    "verification_p_C",
]

mini_df[display_columns]

,question_id,question,reference_answer,generated_answer,automatic_alias_match,answer_format_compliant,mean_logprob,geometric_mean_probability,self_confidence_value,verification_label,verification_p_A,verification_p_B,verification_p_C
0,Q01,Türkiye'nin başkenti neresidir?,Ankara,Ankara.,1,1,-0.014462,0.985642,100,A,1.000000e+00,8.327814e-10,8.864910e-10
1,Q02,Türkiye Cumhuriyeti hangi yıl ilan edildi?,1923,1923.,1,1,-0.121519,0.885574,90,A,9.999993e-01,7.112437e-07,1.780564e-08
2,Q03,Dünya'nın doğal uydusunun adı nedir?,Ay,Bilmiyorum.,0,1,-0.000431,0.999570,50,B,6.796959e-10,9.999993e-01,7.112437e-07
3,Q04,Ay'a ilk ayak basan insan kimdir?,Neil Armstrong,Neil Armstrong.,1,1,-0.035184,0.965428,85,A,1.000000e+00,1.522998e-08,1.323203e-08
4,Q05,Suyun kimyasal formülü nedir?,H2O,H₂O,1,1,-0.001190,0.998811,95,A,1.000000e+00,1.257312e-10,3.638152e-10


In [ ]:
print(
    "Otomatik alias accuracy:",
    mini_df["automatic_alias_match"].mean(),
)

print(
    "Format uyumu:",
    mini_df["answer_format_compliant"].mean(),
)

print(
    "Confidence parse oranı:",
    mini_df["confidence_parse_success"].mean(),
)

print(
    "Verification parse oranı:",
    mini_df["verification_parse_success"].mean(),
)

print(
    "Ortalama self-confidence:",
    mini_df["self_confidence_value"].mean(),
)

print(
    "Ortalama mean logprob:",
    mini_df["mean_logprob"].mean(),
)

Otomatik alias accuracy: 0.8
Format uyumu: 1.0
Confidence parse oranı: 1.0
Verification parse oranı: 1.0
Ortalama self-confidence: 84.0
Ortalama mean logprob: -0.034557023773208524


In [ ]:
import os
import pandas as pd

FULL_PILOT_PATH = "/content/qwen3_4b_turkish_confidence_pilot20.csv"

# İlk 5 sonuç mevcut dosyadan alınır.
if os.path.exists(MINI_PILOT_PATH):
    completed_df = pd.read_csv(MINI_PILOT_PATH)
    full_results = completed_df.to_dict("records")
else:
    full_results = []

completed_ids = {
    row["question_id"]
    for row in full_results
}

print("Daha önce tamamlanan sorular:", sorted(completed_ids))
print("Kalan soru sayısı:", len(pilot_questions) - len(completed_ids))

Daha önce tamamlanan sorular: ['Q01', 'Q02', 'Q03', 'Q04', 'Q05']
Kalan soru sayısı: 15


In [ ]:
for item in pilot_questions:
    question_id = item["question_id"]

    if question_id in completed_ids:
        continue

    print(
        f"Çalıştırılıyor: {question_id} — "
        f"{item['question']}"
    )

    try:
        result = run_single_pilot_question(item)
        full_results.append(result)
        completed_ids.add(question_id)

        current_df = pd.DataFrame(full_results)

        current_df.to_csv(
            FULL_PILOT_PATH,
            index=False,
            encoding="utf-8-sig",
        )

        print("  Cevap:", repr(result["generated_answer"]))
        print("  Referans:", repr(result["reference_answer"]))
        print("  Alias eşleşmesi:", result["automatic_alias_match"])
        print("  Mean logprob:", round(result["mean_logprob"], 6))
        print("  Self-confidence:", result["self_confidence_value"])
        print("  Verification:", result["verification_label"])
        print("  Süre:", round(result["elapsed_seconds"], 2), "sn")
        print()

    except Exception as error:
        print(
            f"HATA — {question_id}: "
            f"{type(error).__name__}: {error}"
        )

        if full_results:
            pd.DataFrame(full_results).to_csv(
                FULL_PILOT_PATH,
                index=False,
                encoding="utf-8-sig",
            )

        raise

print("20 soruluk pilot tamamlandı.")
print("CSV:", FULL_PILOT_PATH)

Çalıştırılıyor: Q06 — İstiklal Marşı'nın yazarı kimdir?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


  Cevap: 'Mustafa Kemal Atatürk.'
  Referans: 'Mehmet Akif Ersoy'
  Alias eşleşmesi: 0
  Mean logprob: -0.161409
  Self-confidence: 85
  Verification: B
  Süre: 2.85 sn

Çalıştırılıyor: Q07 — Bir üçgenin iç açılarının toplamı kaç derecedir?
  Cevap: '180 derece.'
  Referans: '180'
  Alias eşleşmesi: 1
  Mean logprob: -0.007264
  Self-confidence: 90
  Verification: A
  Süre: 1.57 sn

Çalıştırılıyor: Q08 — Güneş Sistemi'nin en büyük gezegeni hangisidir?
  Cevap: 'Jüpiter'
  Referans: 'Jüpiter'
  Alias eşleşmesi: 1
  Mean logprob: -0.093195
  Self-confidence: 90
  Verification: A
  Süre: 1.25 sn

Çalıştırılıyor: Q09 — Türkiye'nin en kalabalık şehri hangisidir?
  Cevap: 'İstanbul'
  Referans: 'İstanbul'
  Alias eşleşmesi: 1
  Mean logprob: -0.215121
  Self-confidence: 70
  Verification: A
  Süre: 1.12 sn

Çalıştırılıyor: Q10 — DNA'nın Türkçe açılımı nedir?
  Cevap: 'DNA: Deoksiribonükleik Asit.'
  Referans: 'Deoksiribonükleik asit'
  Alias eşleşmesi: 0
  Mean logprob: -0.058451
  Self-conf

In [ ]:
full_df = pd.read_csv(FULL_PILOT_PATH)

display_columns = [
    "question_id",
    "question",
    "reference_answer",
    "generated_answer",
    "automatic_alias_match",
    "mean_logprob",
    "geometric_mean_probability",
    "self_confidence_value",
    "verification_label",
    "verification_p_A",
    "verification_p_B",
    "verification_p_C",
]

full_df[display_columns]

,question_id,question,reference_answer,generated_answer,automatic_alias_match,mean_logprob,geometric_mean_probability,self_confidence_value,verification_label,verification_p_A,verification_p_B,verification_p_C
0,Q01,Türkiye'nin başkenti neresidir?,Ankara,Ankara.,1,-0.014462,0.985642,100,A,1.000000e+00,8.327814e-10,8.864910e-10
1,Q02,Türkiye Cumhuriyeti hangi yıl ilan edildi?,1923,1923.,1,-0.121519,0.885574,90,A,9.999993e-01,7.112437e-07,1.780564e-08
2,Q03,Dünya'nın doğal uydusunun adı nedir?,Ay,Bilmiyorum.,0,-0.000431,0.999570,50,B,6.796959e-10,9.999993e-01,7.112437e-07
3,Q04,Ay'a ilk ayak basan insan kimdir?,Neil Armstrong,Neil Armstrong.,1,-0.035184,0.965428,85,A,1.000000e+00,1.522998e-08,1.323203e-08
4,Q05,Suyun kimyasal formülü nedir?,H2O,H₂O,1,-0.001190,0.998811,95,A,1.000000e+00,1.257312e-10,3.638152e-10
5,Q06,İstiklal Marşı'nın yazarı kimdir?,Mehmet Akif Ersoy,Mustafa Kemal Atatürk.,0,-0.161409,0.850944,85,B,2.873677e-07,9.999967e-01,2.994440e-06
6,Q07,Bir üçgenin iç açılarının toplamı kaç derecedir?,180,180 derece.,1,-0.007264,0.992762,90,A,1.000000e+00,2.500463e-10,3.995728e-10
7,Q08,Güneş Sistemi'nin en büyük gezegeni hangisidir?,Jüpiter,Jüpiter,1,-0.093195,0.911016,90,A,1.000000e+00,1.925250e-08,1.030512e-08
8,Q09,Türkiye'nin en kalabalık şehri hangisidir?,İstanbul,İstanbul,1,-0.215121,0.806443,70,A,9.998785e-01,1.159187e-04,5.593689e-06
9,Q10,DNA'nın Türkçe açılımı nedir?,Deoksiribonükleik asit,DNA: Deoksiribonükleik Asit.,0,-0.058451,0.943225,95,A,1.000000e+00,1.699027e-08,1.725783e-08


In [ ]:
print("Toplam kayıt:", len(full_df))

print(
    "Otomatik alias accuracy:",
    round(full_df["automatic_alias_match"].mean(), 3),
)

print(
    "Format uyumu:",
    round(full_df["answer_format_compliant"].mean(), 3),
)

print(
    "Confidence parse oranı:",
    round(full_df["confidence_parse_success"].mean(), 3),
)

print(
    "Verification parse oranı:",
    round(full_df["verification_parse_success"].mean(), 3),
)

print(
    "Ortalama self-confidence:",
    round(full_df["self_confidence_value"].mean(), 2),
)

print(
    "Ortalama mean logprob:",
    round(full_df["mean_logprob"].mean(), 6),
)

print("\nVerification dağılımı:")
print(full_df["verification_label"].value_counts(dropna=False))

Toplam kayıt: 20
Otomatik alias accuracy: 0.6
Format uyumu: 1.0
Confidence parse oranı: 1.0
Verification parse oranı: 1.0
Ortalama self-confidence: 82.0
Ortalama mean logprob: -0.096983

Verification dağılımı:
verification_label
A    15
B     5
Name: count, dtype: int64


In [ ]:
manual_labels = {
    "Q01": (1, "NONE", ""),
    "Q02": (1, "NONE", ""),
    "Q03": (
        0,
        "UNKNOWN_RESPONSE",
        "Model kolay bir soruda Bilmiyorum cevabı verdi.",
    ),
    "Q04": (1, "NONE", ""),
    "Q05": (1, "NONE", ""),
    "Q06": (
        0,
        "FACTUAL_ERROR",
        "Doğru cevap Mehmet Akif Ersoy.",
    ),
    "Q07": (1, "NONE", ""),
    "Q08": (1, "NONE", ""),
    "Q09": (1, "NONE", ""),
    "Q10": (
        1,
        "NONE",
        "Cevap doğru; otomatik alias eşleşmesi biçim nedeniyle başarısız.",
    ),
    "Q11": (
        0,
        "FACTUAL_ERROR",
        "Doğru cevap Osman Gazi; model Mehmed II cevabını verdi.",
    ),
    "Q12": (
        1,
        "NONE",
        "Cevap doğru olmasına rağmen self-verification B üretti.",
    ),
    "Q13": (1, "NONE", ""),
    "Q14": (
        0,
        "FACTUAL_ERROR",
        "Doğru cevap Fyodor Dostoyevski.",
    ),
    "Q15": (
        1,
        "NONE",
        "Yaklaşık 300.000 km/s cevabı doğru.",
    ),
    "Q16": (1, "NONE", ""),
    "Q17": (
        1,
        "NONE",
        "Cevap doğru; ek TCP öneki nedeniyle alias eşleşmedi.",
    ),
    "Q18": (1, "NONE", ""),
    "Q19": (1, "NONE", ""),
    "Q20": (
        0,
        "UNKNOWN_RESPONSE",
        "Doğru cevap deri/cilt; model Bilmiyorum dedi.",
    ),
}

full_df["manual_is_correct"] = full_df["question_id"].map(
    lambda qid: manual_labels[qid][0]
)

full_df["manual_error_type"] = full_df["question_id"].map(
    lambda qid: manual_labels[qid][1]
)

full_df["manual_notes"] = full_df["question_id"].map(
    lambda qid: manual_labels[qid][2]
)

LABELED_PILOT_PATH = (
    "/content/qwen3_4b_turkish_confidence_pilot20_labeled.csv"
)

full_df.to_csv(
    LABELED_PILOT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Kaydedildi:", LABELED_PILOT_PATH)
print("Manuel accuracy:", full_df["manual_is_correct"].mean())

Kaydedildi: /content/qwen3_4b_turkish_confidence_pilot20_labeled.csv
Manuel accuracy: 0.75


In [ ]:
correct_df = full_df[full_df["manual_is_correct"] == 1]
incorrect_df = full_df[full_df["manual_is_correct"] == 0]

full_df["verification_predicted_correct"] = (
    full_df["verification_label"] == "A"
).astype(int)

full_df["verification_is_correct"] = (
    full_df["verification_predicted_correct"]
    == full_df["manual_is_correct"]
).astype(int)

print("Toplam soru:", len(full_df))
print("Manuel accuracy:", full_df["manual_is_correct"].mean())

print(
    "Doğru cevap ortalama self-confidence:",
    correct_df["self_confidence_value"].mean(),
)

print(
    "Yanlış cevap ortalama self-confidence:",
    incorrect_df["self_confidence_value"].mean(),
)

print(
    "Doğru cevap ortalama mean logprob:",
    correct_df["mean_logprob"].mean(),
)

print(
    "Yanlış cevap ortalama mean logprob:",
    incorrect_df["mean_logprob"].mean(),
)

print(
    "Self-verification doğruluğu:",
    full_df["verification_is_correct"].mean(),
)

print("\nHata türleri:")
print(full_df["manual_error_type"].value_counts())

Toplam soru: 20
Manuel accuracy: 0.75
Doğru cevap ortalama self-confidence: 88.33333333333333
Yanlış cevap ortalama self-confidence: 63.0
Doğru cevap ortalama mean logprob: -0.07190294094457478
Yanlış cevap ortalama mean logprob: -0.1722248158092194
Self-verification doğruluğu: 0.9

Hata türleri:
manual_error_type
NONE                15
FACTUAL_ERROR        3
UNKNOWN_RESPONSE     2
Name: count, dtype: int64


# Self-Consistency Testi

In [ ]:
import random
import numpy as np
import torch


@torch.inference_mode()
def generate_sampled_answer(
    question: str,
    seed: int,
    max_new_tokens: int = 20,
    temperature: float = 0.7,
    top_p: float = 0.9,
) -> str:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    messages = build_answer_messages(question)

    prompt_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(
        prompt_text,
        return_tensors="pt",
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    input_length = inputs["input_ids"].shape[1]
    generated_ids = outputs[0, input_length:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

In [ ]:
from collections import Counter


def compute_self_consistency(
    question: str,
    seeds: list[int],
) -> dict:
    sampled_answers = []

    for seed in seeds:
        answer = generate_sampled_answer(
            question=question,
            seed=seed,
        )

        sampled_answers.append(answer)

    normalized_answers = [
        normalize_text(answer)
        for answer in sampled_answers
    ]

    counts = Counter(normalized_answers)

    majority_normalized_answer, majority_count = (
        counts.most_common(1)[0]
    )

    majority_consistency = (
        majority_count / len(sampled_answers)
    )

    unique_answer_count = len(counts)

    return {
        "sampled_answers": sampled_answers,
        "normalized_answers": normalized_answers,
        "answer_counts": dict(counts),
        "majority_normalized_answer": majority_normalized_answer,
        "majority_count": majority_count,
        "self_consistency": majority_consistency,
        "unique_answer_count": unique_answer_count,
    }

In [ ]:
SELF_CONSISTENCY_SEEDS = [
    101,
    202,
    303,
    404,
    505,
]

q01_consistency = compute_self_consistency(
    question=pilot_questions[0]["question"],
    seeds=SELF_CONSISTENCY_SEEDS,
)

print("Soru:", pilot_questions[0]["question"])
print("Örnek cevaplar:")

for index, answer in enumerate(
    q01_consistency["sampled_answers"],
    start=1,
):
    print(f"{index}. {repr(answer)}")

print(
    "Cevap dağılımı:",
    q01_consistency["answer_counts"],
)

print(
    "Self-consistency:",
    q01_consistency["self_consistency"],
)

print(
    "Benzersiz cevap sayısı:",
    q01_consistency["unique_answer_count"],
)

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Soru: Türkiye'nin başkenti neresidir?
Örnek cevaplar:
1. 'Ankara.'
2. 'Ankara.'
3. 'Ankara.'
4. 'Ankara.'
5. 'Ankara.'
Cevap dağılımı: {'ankara': 5}
Self-consistency: 1.0
Benzersiz cevap sayısı: 1


In [ ]:
critical_question_ids = [
    "Q03",
    "Q11",
    "Q14",
]

critical_items = {
    item["question_id"]: item
    for item in pilot_questions
    if item["question_id"] in critical_question_ids
}

critical_consistency_results = []

for question_id in critical_question_ids:
    item = critical_items[question_id]

    result = compute_self_consistency(
        question=item["question"],
        seeds=SELF_CONSISTENCY_SEEDS,
    )

    critical_consistency_results.append(
        {
            "question_id": question_id,
            "question": item["question"],
            "reference_answer": item["reference_answer"],
            "sampled_answers": result["sampled_answers"],
            "answer_counts": result["answer_counts"],
            "self_consistency": result["self_consistency"],
            "unique_answer_count": result["unique_answer_count"],
        }
    )

for result in critical_consistency_results:
    print("=" * 80)
    print(result["question_id"], result["question"])
    print("Referans:", result["reference_answer"])
    print("Cevaplar:")

    for answer in result["sampled_answers"]:
        print("-", repr(answer))

    print("Dağılım:", result["answer_counts"])
    print(
        "Self-consistency:",
        result["self_consistency"],
    )

Q03 Dünya'nın doğal uydusunun adı nedir?
Referans: Ay
Cevaplar:
- 'Bilmiyorum.'
- 'Bilmiyorum.'
- 'Bilmiyorum.'
- 'Bilmiyorum.'
- 'Bilmiyorum.'
Dağılım: {'bilmiyorum': 5}
Self-consistency: 1.0
Q11 Osmanlı Devleti'nin kurucusu kimdir?
Referans: Osman Gazi
Cevaplar:
- 'Mehmed II.'
- 'Mehmed II.'
- 'Mustafa Kemal Atatürk.'
- 'Mehmed II.'
- 'Mehmed II.'
Dağılım: {'mehmed ii': 4, 'mustafa kemal atatürk': 1}
Self-consistency: 0.8
Q14 Suç ve Ceza romanının yazarı kimdir?
Referans: Fyodor Dostoyevski
Cevaplar:
- 'Ahmet Muzaffer.'
- 'Ahmet Muhtar.'
- 'Ahmet Mümtaz.'
- 'Ahmet Muhtar.'
- 'Bilmiyorum.'
Dağılım: {'ahmet muzaffer': 1, 'ahmet muhtar': 2, 'ahmet mümtaz': 1, 'bilmiyorum': 1}
Self-consistency: 0.4
